<a href="https://colab.research.google.com/github/sergiojsp/gnn-practicas/blob/main/gnnCora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Ejemplo de una red neuronal basada en grafos (GNN) utilizando la biblioteca **PyTorch Geometric (PyG)**.

El siguiente código define un modelo de GNN y lo entrena para clasificar nodos utilizando el dataset **Cora**, un grafo de citas de artículos científicos donde los nodos representan los artículos y las aristas las citas entre ellos. La tarea consiste en predecir la categoría de cada artículo.

In [ ]:
pip install torch-geometric pyvis

# Importar librerías y preparar datos

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.datasets import Planetoid
from torch_geometric.utils import index_to_mask
from torch_geometric.utils import to_networkx
import matplotlib.pyplot as plt
import networkx as nx
import random

import torch
from torch_geometric.utils import index_to_mask


num_runs = 10        # Número de corridas para promedio
hidden_channels = 16 # capas ocultas
dropout = 0.5
lr = 0.01
weight_decay = 5e-4
max_epochs = 400
patience = 10        # número de epocas a esperar para detener la predicción

# 3. Función de split fijo (igual que Kipf & Welling)
def fixed_split(data, num_train=140, num_val=500, num_test=1000):
    num_nodes = data.num_nodes
    indices = torch.randperm(num_nodes)
    train_idx = indices[:num_train]
    val_idx = indices[num_train:num_train+num_val]
    test_idx = indices[num_train+num_val:num_train+num_val+num_test]
    data.train_mask = index_to_mask(train_idx, size=num_nodes)
    data.val_mask = index_to_mask(val_idx, size=num_nodes)
    data.test_mask = index_to_mask(test_idx, size=num_nodes)
    return data

# Cargar el dataset
dataset = Planetoid(root='/tmp/Cora', name='Cora')
data = dataset[0]
data = fixed_split(data, num_train=140, num_val=500, num_test=1000)
print(data)

print(f'Nodos de entrenamiento: {data.train_mask.sum().item()}')
print(f'Nodos de validación: {data.val_mask.sum().item()}')
print(f'Nodos de prueba: {data.test_mask.sum().item()}')



# Arquitectura de la GNN

In [ ]:
# Definir el modelo GCN
class GNN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(GNN, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

# Inicialización
model = GNN(data.num_node_features, hidden_channels, out_channels=dataset.num_classes)
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
criterion = torch.nn.CrossEntropyLoss()

# Funciones de entrenamiento, validación y test

In [ ]:
# Función de entrenamiento
def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

# Función de validación
def validate():
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        loss = criterion(out[data.val_mask], data.y[data.val_mask])
        pred = out.argmax(dim=1)
        correct = (pred[data.val_mask] == data.y[data.val_mask]).sum()
        acc = int(correct) / int(data.val_mask.sum())
    return loss.item(), acc

# Función de test
def test():
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        pred = out.argmax(dim=1)
        correct = (pred[data.test_mask] == data.y[data.test_mask]).sum()
        acc = int(correct) / int(data.test_mask.sum())
    return acc

# Entrenamiento con early stopping

In [ ]:
# Entrenamiento + Validación + Early Stopping
best_val_loss = float('inf')
counter = 0

for epoch in range(1, max_epochs):
    loss = train()
    val_loss, val_acc = validate()
    #test_acc = test()

    print(f'Época: {epoch:03d}, Train Loss: {loss:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        torch.save(model.state_dict(), 'mejor_modelo.pt')  # Guarda el mejor modelo
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping aplicado.")
            break

# 9. Evaluación final del mejor modelo
model.load_state_dict(torch.load('mejor_modelo.pt'))
final_test_acc = test()
print(f'\n📌 Precisión del modelo: {final_test_acc:.4f}')

In [ ]:
def show_predictions_text(data, model, class_names):
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        pred = out.argmax(dim=1)

        test_indices = data.test_mask.nonzero(as_tuple=True)[0]
        true_labels = data.y[test_indices]
        pred_labels = pred[test_indices]

        print("\nPredicciones en nodos de prueba:")
        print(f"{'Num':<4}{'Nodo':<6}{'Verdad':<25}{'Predicción':<25}{'Correcto?'}")
        print("-" * 65)
        for i in range(len(test_indices)):
            node_id = int(test_indices[i])
            true_class = class_names[int(true_labels[i])]
            pred_class = class_names[int(pred_labels[i])]
            correct = "✅" if true_class == pred_class else "❌"
            print(f"{i:<4}{node_id:<6}{true_class:<25}{pred_class:<25}{correct}")


class_names = [
    "Case_Based",
    "Genetic_Algorithms",
    "Neural_Networks",
    "Probabilistic_Methods",
    "Reinforcement_Learning",
    "Rule_Learning",
    "Theory"
]

show_predictions_text(data, model, class_names)

# Mostrar grafo con networkx

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.patches import Patch

# Nombres de las clases del dataset Cora
class_names = [
    "Case_Based",              # 0
    "Genetic_Algorithms",      # 1
    "Neural_Networks",         # 2
    "Probabilistic_Methods",   # 3
    "Reinforcement_Learning",  # 4
    "Rule_Learning",           # 5
    "Theory"                   # 6
]
# Convertir a NetworkX
G = to_networkx(data, to_undirected=True)

# Layout: cómo se posicionan los nodos
pos = nx.spring_layout(G, seed=42)

# Obtener las etiquetas de clase como numpy array para mapear colores
colors = data.y.numpy()

# Mapa de colores (el Set1 tiene 9 colores, aquí solo usamos 7 para las clases de Cora)
cmap = plt.cm.Set1

plt.figure(figsize=(12, 8))
nx.draw(G, pos, node_size=20, with_labels=False, edge_color="gray", alpha=0.7, node_color=colors, cmap=cmap)

# Crear leyenda manualmente
legend_elements = [
    Patch(facecolor=cmap(i / len(class_names)), label=class_names[i])
    for i in range(len(class_names))
]

plt.legend(handles=legend_elements, title="Clases", loc="best")
plt.title("Grafo del dataset Cora")
plt.show()


-----

### Explicación del código

  * **Carga de datos:** El código utiliza `Planetoid` de PyG para descargar y cargar el conjunto de datos de Cora. El objeto `data` contiene la estructura del grafo (`edge_index`) y las características de los nodos (`x`). `train_mask`, `val_mask` y `test_mask` son máscaras booleanas que se usan para dividir los nodos en conjuntos de entrenamiento, validación y prueba.
  * **Definición de la GNN:** La clase `GNN` hereda de `torch.nn.Module`. Usa la capa **`GCNConv`**, que implementa una convolución de grafos. Esta capa agrega las características de los nodos vecinos y las combina con las características del nodo central para aprender una nueva representación. La función `forward` describe el flujo de datos a través de las dos capas de la GNN.
  * **Entrenamiento:** El bucle de entrenamiento llama a la función `train()`, que realiza los pasos típicos de un ciclo de aprendizaje supervisado: propagación hacia adelante para obtener las predicciones, cálculo de la pérdida, retropropagación para calcular los gradientes y actualización de los pesos.
  * **Evaluación:** La función `test()` se usa para evaluar el rendimiento del modelo en los datos de prueba.

Este ejemplo muestra la simplicidad de construir una GNN básica con PyTorch Geometric, que maneja la complejidad de la propagación de mensajes entre los nodos del grafo de forma eficiente.